# MacroVision v1 mass fine-tuning on Kaggle
This notebook trains the shared RGB + explicit-geometry mass model. Enable a GPU and attach Nutrition5k plus a Kaggle dataset containing this repository.

In [ ]:
from pathlib import Path
import os, sys, torch

# EDIT THESE PATHS after attaching the datasets in Kaggle.
REPO_BACKEND = Path('/kaggle/input/macrovision-source/backend')
NUTRITION5K = Path('/kaggle/input/nutrition5k')
METADATA = NUTRITION5K / 'metadata' / 'dish_metadata_cafe1.csv'
GEOMETRY_CSV = None  # Set to a precomputed geometry CSV for Tier 1/2 training.
WORK = Path('/kaggle/working/macrovision_v1')
WORK.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_BACKEND))
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Build a dish-level split manifest
The preparation script recursively locates RGB files and uses a stable dish-id split, preventing frames of the same physical meal from leaking across train and validation.

In [ ]:
from argparse import Namespace
from app.ai.MacroVision.v1.prepare_nutrition5k import main as prepare

manifest = WORK / 'nutrition5k_v1.csv'
prepare(Namespace(dataset_root=str(NUTRITION5K), metadata_csv=str(METADATA), geometry_csv=str(GEOMETRY_CSV) if GEOMETRY_CSV else None, output=str(manifest)))

In [ ]:
import csv
from collections import Counter
with manifest.open() as f:
    rows = list(csv.DictReader(f))
print('rows:', len(rows), 'splits:', Counter(r['split'] for r in rows))
print(rows[0] if rows else 'No rows: verify METADATA and dataset layout')

## Train
First run with `freeze_backbone=True` as a smoke test. For the final run, set it to `False`. Without a geometry CSV this trains only the monocular fallback; it does not train metric RGB-D behavior.

In [ ]:
from app.ai.MacroVision.v1.train import train
args = Namespace(
    manifest=str(manifest), data_root=str(NUTRITION5K), output_dir=str(WORK / 'checkpoints'),
    epochs=30, batch_size=64, workers=2, lr=1e-4, weight_decay=1e-4,
    volume_weight=0.25, seed=42, device='cuda' if torch.cuda.is_available() else 'cpu',
    freeze_backbone=False, no_pretrained=False,
)
best_checkpoint = train(args)
print('Best checkpoint:', best_checkpoint)

In [ ]:
import json
history = json.loads((WORK / 'checkpoints' / 'history.json').read_text())
best = min(history, key=lambda x: x['val']['mass_mae_g'])
best

## Export
Download `macrovision_v1_best.pt` from the Kaggle output and place it under `MacroVision/v1/weights/` (or pass its path explicitly). Record the test-set MAE/MAPE before integrating it into the API.